In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import geopandas as gpd
import glob
import os
import matplotlib.pyplot as plt

In [ ]:
# ---------------- CONFIG ----------------
folder = "/content/drive/MyDrive/Congestion_LISA"
files = glob.glob(os.path.join(folder, "*.geojson")) + glob.glob(os.path.join(folder, "*.json"))
print(f"Found {len(files)} files")

dfs = []

# ---------------- LOAD SNAPSHOTS ----------------
for f in files:
    try:
        gdf = gpd.read_file(f)

        # Parse timestamp from filename
        # Example: congestion_lisa_2026-03-17_07_00_00
        base = os.path.splitext(os.path.basename(f))[0]
        stamp = base.replace("congestion_lisa_", "")

        try:
            dt = pd.to_datetime(stamp, format="%Y-%m-%d_%H_%M_%S")
        except:
            dt = pd.NaT

        gdf["snapshot_file"] = base
        gdf["Date"] = dt

        dfs.append(gdf)

    except Exception as e:
        print(f"Skipping {f}: {e}")

combined = pd.concat(dfs, ignore_index=True)
print(f"Combined rows: {len(combined)}")
print("Columns found:")
print(combined.columns.tolist())

# ---------------- IDENTIFY REGION FIELD ----------------
region_candidates = ["region", "Region", "city", "City"]
region_field = next((c for c in region_candidates if c in combined.columns), None)

if region_field is None:
    raise ValueError(f"No city/region field found. Columns available: {combined.columns.tolist()}")

combined["Region"] = combined[region_field].astype(str).str.replace("_", " ", regex=False)

# ---------------- CLEAN LABEL FIELD ----------------
if "congestionLabel" not in combined.columns:
    raise ValueError(f"'congestionLabel' not found. Columns available: {combined.columns.tolist()}")

combined["congestionLabel"] = combined["congestionLabel"].astype(str).str.strip()

valid_labels = ["Free Flowing", "Moderate Flow", "Congested"]
combined = combined[combined["congestionLabel"].isin(valid_labels)].copy()

# ---------------- CLASSIFY AM / PM ----------------
combined["Hour"] = combined["Date"].dt.hour

def classify_period(hour):
    if pd.isna(hour):
        return None
    if hour in [7, 8, 9]:
        return "AM"
    elif hour in [16, 17, 18]:
        return "PM"
    else:
        return None

combined["Travel Period"] = combined["Hour"].apply(classify_period)

# ---------------- SUMMARY BY CITY ----------------
city_summary = (
    combined.groupby(["Region", "congestionLabel"])
    .size()
    .reset_index(name="Count")
)

city_pivot = city_summary.pivot(
    index="Region",
    columns="congestionLabel",
    values="Count"
).fillna(0)

for label in valid_labels:
    if label not in city_pivot.columns:
        city_pivot[label] = 0

city_pivot = city_pivot[valid_labels]
city_pivot["Total"] = city_pivot.sum(axis=1)

for label in valid_labels:
    city_pivot[f"{label} (%)"] = (city_pivot[label] / city_pivot["Total"] * 100).round(1)

city_pivot = city_pivot.reset_index().rename(columns={"Region": "City"})
city_pivot = city_pivot.sort_values("City").reset_index(drop=True)

print("\nCluster summary by city:")
print(city_pivot)

# ---------------- SUMMARY BY CITY + TRAVEL PERIOD ----------------
period_summary = (
    combined.dropna(subset=["Travel Period"])
    .groupby(["Region", "Travel Period", "congestionLabel"])
    .size()
    .reset_index(name="Count")
)

period_pivot = period_summary.pivot_table(
    index=["Region", "Travel Period"],
    columns="congestionLabel",
    values="Count",
    fill_value=0
)

for label in valid_labels:
    if label not in period_pivot.columns:
        period_pivot[label] = 0

period_pivot = period_pivot[valid_labels]
period_pivot["Total"] = period_pivot.sum(axis=1)

for label in valid_labels:
    period_pivot[f"{label} (%)"] = (period_pivot[label] / period_pivot["Total"] * 100).round(1)

period_pivot = period_pivot.reset_index().rename(columns={"Region": "City"})
period_pivot = period_pivot.sort_values(["City", "Travel Period"]).reset_index(drop=True)

print("\nCluster summary by city and travel period:")
print(period_pivot)

# ---------------- SAVE OUTPUTS ----------------
city_out = "/content/drive/MyDrive/cluster_summary_by_city.csv"
period_out = "/content/drive/MyDrive/cluster_summary_by_city_period.csv"

city_pivot.to_csv(city_out, index=False)
period_pivot.to_csv(period_out, index=False)

print(f"\nSaved:\n{city_out}\n{period_out}")

# ---------------- COLORS ----------------
cluster_colors = {
    "Free Flowing": "#2E8B57",   # green
    "Moderate Flow": "#F4A300",  # orange
    "Congested": "#C0392B"       # red
}

color_list = [cluster_colors[label] for label in valid_labels]

# ---------------- BAR CHART: CITY COUNTS ----------------
city_counts = city_pivot.set_index("City")[valid_labels]
ax = city_counts.plot(kind="bar", figsize=(12, 6), color=color_list)

plt.title("Congestion Class Counts by City", fontsize=14)
plt.xlabel("City", fontsize=12)
plt.ylabel("Number of Segments", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.legend(title="Congestion Class", frameon=False)
plt.tight_layout()
plt.show()

# ---------------- BAR CHART: CITY PERCENTAGES ----------------
city_pct_cols = [f"{label} (%)" for label in valid_labels]
city_pct = city_pivot.set_index("City")[city_pct_cols]
city_pct.columns = valid_labels  # cleaner legend labels

ax = city_pct.plot(kind="bar", figsize=(12, 6), color=color_list)

plt.title("Congestion Class Percentages by City", fontsize=14)
plt.xlabel("City", fontsize=12)
plt.ylabel("Percent of Segments", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.legend(title="Congestion Class", frameon=False)
plt.tight_layout()
plt.show()

# ---------------- BAR CHART: CITY + PERIOD COUNTS ----------------
period_pivot["City_Period"] = period_pivot["City"] + " (" + period_pivot["Travel Period"] + ")"
period_counts = period_pivot.set_index("City_Period")[valid_labels]

ax = period_counts.plot(kind="bar", figsize=(14, 6), color=color_list)

plt.title("Congestion Class Counts by City and Travel Period", fontsize=14)
plt.xlabel("City / Travel Period", fontsize=12)
plt.ylabel("Number of Segments", fontsize=12)
plt.xticks(rotation=60, ha="right")
plt.legend(title="Congestion Class", frameon=False)
plt.tight_layout()
plt.show()

# ---------------- BAR CHART: CITY + PERIOD PERCENTAGES ----------------
period_pct_cols = [f"{label} (%)" for label in valid_labels]
period_pct = period_pivot.set_index("City_Period")[period_pct_cols]
period_pct.columns = valid_labels  # cleaner legend labels

ax = period_pct.plot(kind="bar", figsize=(14, 6), color=color_list)

plt.title("Congestion Class Percentages by City and Travel Period", fontsize=14)
plt.xlabel("City / Travel Period", fontsize=12)
plt.ylabel("Percent of Segments", fontsize=12)
plt.xticks(rotation=60, ha="right")
plt.legend(title="Congestion Class", frameon=False)
plt.tight_layout()
plt.show()

# ---------------- STACKED PERCENT BAR: CITY ----------------
city_pct = city_pivot.set_index("City")[[f"{label} (%)" for label in valid_labels]]
city_pct.columns = valid_labels

ax = city_pct.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6),
    color=color_list,
    width=0.8
)

plt.title("Congestion Class Composition by City", fontsize=14)
plt.xlabel("City", fontsize=12)
plt.ylabel("Percent of Segments", fontsize=12)
plt.xticks(rotation=45, ha="right")
plt.legend(title="Congestion Class", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

# ---------------- STACKED PERCENT BAR: CITY + PERIOD ----------------
period_pct = period_pivot.set_index("City_Period")[[f"{label} (%)" for label in valid_labels]]
period_pct.columns = valid_labels

ax = period_pct.plot(
    kind="bar",
    stacked=True,
    figsize=(14, 6),
    color=color_list,
    width=0.8
)

plt.title("Congestion Class Composition by City and Travel Period", fontsize=14)
plt.xlabel("City / Travel Period", fontsize=12)
plt.ylabel("Percent of Segments", fontsize=12)
plt.xticks(rotation=60, ha="right")
plt.legend(title="Congestion Class", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()